# Geometric Knowledge Network Demo

This notebook demonstrates a lightweight **Geometric Knowledge Network (GKN)** as an enhancement to vector-based retrieval in a RAG workflow.

The notebook is intentionally **code-light**: heavy logic lives in reusable modules under `src/`.

## What this demo shows

- loading a small sample corpus
- chunking the documents
- running baseline vector retrieval
- building a lightweight knowledge network
- running hybrid retrieval with graph expansion
- comparing retrieval outputs
- evaluating against a small benchmark query set


## 1. Setup

This step imports the project modules, loads configuration, and defines a few helper display functions for a cleaner notebook experience.

In [ ]:
from pathlib import Path
import sys
from collections import Counter

sys.path.append(str(Path.cwd().parent / 'src'))

from geometric_knowledge_network.config import GKNConfig
from geometric_knowledge_network.evaluation import evaluate_retrieval, load_query_test_cases
from geometric_knowledge_network.extraction import ConceptExtractor
from geometric_knowledge_network.graph_builder import KnowledgeNetworkBuilder
from geometric_knowledge_network.hybrid_retriever import HybridRetriever
from geometric_knowledge_network.ingest import DocumentIngestor
from geometric_knowledge_network.vector_store import SimpleVectorStore
from geometric_knowledge_network.visualization import draw_subgraph

config = GKNConfig()

def status(message: str):
    print(f'[DONE] {message}')

def preview_results(results, title):
    print(f'\n{title}')
    print('-' * len(title))
    for i, result in enumerate(results, start=1):
        source = getattr(result, 'source', 'vector')
        vector_score = getattr(result, 'vector_score', getattr(result, 'score', 0.0))
        graph_bonus = getattr(result, 'graph_bonus', 0.0)
        final_score = getattr(result, 'final_score', getattr(result, 'score', 0.0))
        preview_text = result.text[:180].replace(chr(10), ' ')
        print(f'{i}. {result.chunk_id} | source={source} | vector={vector_score:.3f} | graph_bonus={graph_bonus:.3f} | final={final_score:.3f}')
        print(f'   {preview_text}...')

status('Setup complete.')

## 2. Load sample documents

The repository includes a small synthetic-but-realistic corpus focused on governance, validation, controls, and incidents. This type of corpus is useful for testing whether graph-aware retrieval helps with traceability and multi-hop context.

In [ ]:
ingestor = DocumentIngestor()
documents = ingestor.load_text_documents(config.sample_docs_dir)
status(f'Loaded {len(documents)} sample documents from {config.sample_docs_dir}.')

for doc in documents:
    print(f'- {doc.doc_id}: {doc.title}')

## 3. Chunk the documents

Documents are split into retrieval units. In this MVP, chunking is simple and local, but still sufficient to test vector retrieval and graph construction.

In [ ]:
chunks = ingestor.chunk_documents(documents, config.chunk_size, config.chunk_overlap)
status(f'Created {len(chunks)} chunks.')

for chunk in chunks[:5]:
    preview_text = chunk.text[:160].replace(chr(10), ' ')
    print(f'{chunk.chunk_id} | doc={chunk.doc_id} | chars={len(chunk.text)}')
    print(f'  {preview_text}...')

## 4. Build the baseline vector retriever

This step builds a simple local vector index over the chunks and runs a baseline semantic retrieval query.

In [ ]:
vector_store = SimpleVectorStore()
vector_store.build(chunks)
status('Vector index built successfully.')

query = 'What evidence is required for validation approval?'
baseline_results = vector_store.search(query, top_k=config.top_k)
preview_results(baseline_results, f'Baseline retrieval results for query: {query}')

## 5. Build the knowledge network

The network connects documents, chunks, and typed heuristic entities such as requirements, controls, evidence, incidents, and generic concepts.

This is still a lightweight MVP, but it is richer than plain concept co-occurrence.

In [ ]:
extractor = ConceptExtractor(config.concept_keywords)
graph = KnowledgeNetworkBuilder(extractor).build(documents, chunks)
status(f'Knowledge network built with {graph.number_of_nodes()} nodes and {graph.number_of_edges()} edges.')

node_type_counts = Counter(graph.nodes[node].get('node_type', 'Unknown') for node in graph.nodes)
for node_type, count in sorted(node_type_counts.items()):
    print(f'- {node_type}: {count}')

## 6. Run hybrid retrieval

Hybrid retrieval starts with vector hits, then expands candidate chunks through the graph. This allows chunks connected through requirements, controls, evidence, or incidents to enter the candidate set even if they were not in the original top-k vector results.

In [ ]:
hybrid = HybridRetriever(vector_store, graph)
hybrid_results = hybrid.search(query, top_k=config.top_k, graph_hops=config.graph_hops)
status('Hybrid retrieval completed.')
preview_results(hybrid_results, f'Hybrid retrieval results for query: {query}')

## 7. Compare baseline and hybrid retrieval

This side-by-side comparison helps illustrate whether graph expansion changes which chunks are returned and how they are ranked.

In [ ]:
print('Baseline chunk IDs:')
for result in baseline_results:
    print(f'  - {result.chunk_id}')

print('\nHybrid chunk IDs:')
for result in hybrid_results:
    print(f'  - {result.chunk_id} (source={result.source})')

## 8. Visualize a local graph neighborhood

A small subgraph around the top hybrid result helps inspect which structured entities and relationships may be influencing retrieval.

In [ ]:
if hybrid_results:
    draw_subgraph(graph, hybrid_results[0].chunk_id, radius=2)
    status(f'Subgraph rendered around {hybrid_results[0].chunk_id}.')
else:
    print('No hybrid results available to visualize.')

## 9. Run the benchmark query set

The repository includes a small benchmark file in `data/eval_queries.json`. We use it here to compute simple retrieval metrics for the current implementation.

In [ ]:
test_cases = load_query_test_cases(config.eval_queries_path)
status(f'Loaded {len(test_cases)} evaluation queries from {config.eval_queries_path}.')

for case in test_cases:
    baseline_case_results = vector_store.search(case.query, top_k=config.top_k)
    hybrid_case_results = hybrid.search(case.query, top_k=config.top_k, graph_hops=config.graph_hops)

    baseline_metrics = evaluate_retrieval(baseline_case_results, case.relevant_chunk_ids)
    hybrid_metrics = evaluate_retrieval(hybrid_case_results, case.relevant_chunk_ids)

    print('\n' + '=' * 100)
    print(f'Query: {case.query}')
    print(f'Category: {case.category}')
    print(f'Baseline metrics: {baseline_metrics}')
    print(f'Hybrid metrics:   {hybrid_metrics}')

## 10. Summary

At this stage, the demo is meant to validate that the pipeline runs end-to-end and that the hybrid retriever behaves differently from the baseline.

The next goal is to strengthen the comparison with richer graph semantics, stronger chunking, and more robust evaluation.